In [0]:
########################################
# Narrated Auto Loader load of orders_bronze for Lesson 06.
#
# The notebook's concepts section has already explained Auto Loader (the
# cloudFiles streaming source), how its checkpoint tracks which files it has
# already processed (exactly-once ingestion), and how trigger(availableNow)
# turns a stream into a one-shot batch that processes everything available
# and stops. This script runs the load while narrating each step.
#
# Includes a deliberate re-run demonstration: after loading the four 2024
# files, the same stream is run again and processes 0 new files, because the
# checkpoint already recorded them.
########################################

try:
    _catalog = my_catalog  # set by Classroom-Setup-Common (reliable)
except NameError:
    _catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
_schema      = "vintage_audio"
_volume_path = f"/Volumes/{_catalog}/{_schema}/data/"
_table       = f"`{_catalog}`.`{_schema}`.orders_bronze"
_checkpoint  = f"/Volumes/{_catalog}/{_schema}/data/_ckpt_orders_bronze"
_schema_loc  = f"/Volumes/{_catalog}/{_schema}/data/_schema_orders_bronze"


def _line(char: str = "-", width: int = 72) -> None:
    print(char * width, flush=True)


def _section(label: str) -> None:
    print(flush=True)
    _line("=")
    print(f"  {label}", flush=True)
    _line("=")


def _say(text: str) -> None:
    print(text, flush=True)


def _table_exists(catalog: str, schema: str, table: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE `{catalog}`.`{schema}`.`{table}`")
        return True
    except Exception:
        return False


def _row_count() -> int:
    return spark.sql(f"SELECT COUNT(*) AS n FROM {_table}").collect()[0]["n"]


def _run_autoloader(glob_pattern: str) -> int:
    """Run a one-shot Auto Loader batch over files matching glob_pattern.
    Returns the number of rows in the table after the run completes."""
    before = _row_count() if _table_exists(_catalog, _schema, "orders_bronze") else 0
    stream = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", _schema_loc)
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("pathGlobFilter", glob_pattern)
            .load(_volume_path)
    )
    query = (
        stream.writeStream
            .option("checkpointLocation", _checkpoint)
            .trigger(availableNow=True)
            .toTable(_table.replace("`", ""))
    )
    query.awaitTermination()
    after = _row_count()
    return after - before


# ----- Prerequisite check -----------------------------------------------------
# This load only needs the four 2024 order CSV files staged in the data volume.
# (It does NOT need the customers/products/stores tables.) Classroom-Setup-Common
# stages these files, so a standalone run of this lesson has them.
import os

def _csv_staged(filename: str) -> bool:
    return os.path.exists(f"{_volume_path}{filename}")

_required_csvs = [f"orders_2024_q{q}.csv" for q in [1, 2, 3, 4]]
_missing = [f for f in _required_csvs if not _csv_staged(f)]

if _missing:
    _line("=")
    _say("  WARNING: the course data files are not staged in your volume.")
    _say(f"  Looked in: {_volume_path}")
    _say("  Run the setup cell at the top of this notebook, or complete the")
    _say("  00 - Course Overview notebook, then re-run this cell.")
    _line("=")
else:
    # Fresh start so the narration is reproducible on re-runs of the demo.
    spark.sql(f"DROP TABLE IF EXISTS {_table}")
    dbutils.fs.rm(_checkpoint, recurse=True)
    dbutils.fs.rm(_schema_loc, recurse=True)

    _section("LOADING orders_bronze WITH AUTO LOADER")
    _say("Now we will run the Auto Loader stream you just read about. The")
    _say("output below narrates each step. Read along.")

    # Step 1: first batch - the 2024 files
    _section("STEP 1 of 2: Ingest the 2024 files")
    _say("Auto Loader points its cloudFiles source at the data volume and")
    _say("processes every matching file in one batch (trigger availableNow),")
    _say("then stops. The target table orders_bronze is created automatically")
    _say("by toTable() if it does not exist.")
    _say("")
    _say("Ingesting orders_2024_q1.csv through orders_2024_q4.csv ...")

    n_first = _run_autoloader("orders_2024_q[1234].csv")

    _say(f"Ingested {n_first:,} rows from the four 2024 files.")
    _say("")
    _say("Behind the scenes, Auto Loader recorded each processed file in its")
    _say("checkpoint. That record is what makes the next step interesting.")

    # Step 2: the re-run demonstration (exactly-once)
    _section("STEP 2 of 2: Re-run the same stream")
    _say("Here is the moment the lesson is built around.")
    _say("")
    _say("We are about to run the EXACT SAME Auto Loader stream again. Same")
    _say("source directory, same glob, same checkpoint. No new files have")
    _say("arrived.")
    _say("")
    _say("What do you expect to happen? Look at the row count below.")
    _say("")
    _say("Re-running the stream ...")

    n_rerun = _run_autoloader("orders_2024_q[1234].csv")

    _say(f"    -> {n_rerun} new rows ingested.")
    _say("")
    _say("Zero rows. The checkpoint already lists all four files, so Auto")
    _say("Loader processed nothing new. This is exactly-once ingestion: the")
    _say("same stream can run on any schedule and only ever picks up files")
    _say("it has not already seen.")

    # Final tally
    _section("LOAD COMPLETE")
    total = _row_count()
    _say(f"orders_bronze is ready with {total:,} total rows across 4 files:")
    _say("")
    _say("  orders_2024_q1.csv  43,188 rows")
    _say("  orders_2024_q2.csv  47,507 rows")
    _say("  orders_2024_q3.csv  51,826 rows")
    _say("  orders_2024_q4.csv  60,463 rows")
    _say("")
    _say("Continue scrolling to verify the load with SELECT queries.")
